In [ ]:
import kagglehub
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path to dataset files:", path)

In [ ]:
!pip install timm -q

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import timm
from tqdm import tqdm

In [ ]:
DATASET = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"

IMG1 = f"{DATASET}/HAM10000_images_part_1"
IMG2 = f"{DATASET}/HAM10000_images_part_2"

META = f"{DATASET}/HAM10000_metadata.csv"

In [ ]:
# Load metadata
df = pd.read_csv(META)

def get_path(img_id):

    p1 = os.path.join(IMG1, img_id + ".jpg")
    p2 = os.path.join(IMG2, img_id + ".jpg")

    if os.path.exists(p1):
        return p1
    return p2

df["path"] = df["image_id"].apply(get_path)

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["dx"])

NUM_CLASSES = df["label"].nunique()

print("Classes:", NUM_CLASSES)

In [ ]:
# Train validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [ ]:
# class imbalance weights
weights = compute_class_weight(
    "balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(weights, dtype=torch.float).cuda()

In [ ]:
# Image transforms
IMG_SIZE = 224

train_tfms = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


val_tfms = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])


In [ ]:
# Dataset
class HAMDataset(Dataset):

    def __init__(self, df, transform=None):

        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(row.path).convert("RGB")
        label = row.label

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
# dataloader
BATCH_SIZE = 32

train_ds = HAMDataset(train_df, train_tfms)
val_ds = HAMDataset(val_df, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# EfficientNet branch
class EfficientNetBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = timm.create_model(
            "efficientnet_b4",
            pretrained=True,
            num_classes=0
        )

    def forward(self, x):

        x = self.model(x)

        return x

In [ ]:
# Swin Transformer branch
class SwinBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

    def forward(self, x):

        x = self.model(x)

        return x

In [ ]:
# Hybrid model
class HybridModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.cnn = EfficientNetBranch()
        self.tr = SwinBranch()

        concat_dim = 1792 + 1024

        self.mlp = nn.Sequential(

            nn.Linear(concat_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        f1 = self.cnn(x)
        f2 = self.tr(x)

        x = torch.cat([f1, f2], dim=1)

        out = self.mlp(x)

        return out

In [ ]:
# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridModel(NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
EPOCHS = 100
PATIENCE = 50

best_acc = 0
counter = 0

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    model.train()
    train_loss = 0

    for imgs, labels in tqdm(train_loader):

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # validation

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for imgs, labels in val_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total

    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_acc:

        best_acc = val_acc
        counter = 0

        torch.save(model.state_dict(), "best_model.pth")

        print("Model saved")

    else:

        counter += 1
        print("Early stop counter:", counter)

        if counter >= PATIENCE:
            print("Early stopping triggered")
            break

In [ ]:
model = HybridModel(NUM_CLASSES)
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize

In [ ]:
model = HybridModel(NUM_CLASSES)

model.load_state_dict(
    torch.load("best_model.pth", map_location=device)
)

model.to(device)

model.eval()

In [ ]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():

    for imgs, labels in val_loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)

        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [ ]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=encoder.classes_
))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

from sklearn.preprocessing import label_binarize
labels_bin = label_binarize(all_labels, classes=np.arange(NUM_CLASSES))

In [ ]:
plt.figure(figsize=(8,6))

for i in range(NUM_CLASSES):

    precision, recall, _ = precision_recall_curve(
        labels_bin[:, i],
        all_probs[:, i]
    )

    ap = average_precision_score(labels_bin[:, i], all_probs[:, i])

    plt.plot(recall, precision, label=f"{encoder.classes_[i]} (AP={ap:.2f})")


plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.legend()

plt.show()